# Layer 2: context-variable ablation

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import load_npz, hstack
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score

ROOT = next((p for p in [Path("/workspace"), Path.cwd(), Path.cwd().parent]
             if (p / "output" / "10_layer2_features").exists()), Path.cwd())
FEAT = ROOT / "output" / "10_layer2_features"
OUT = ROOT / "output" / "14_layer2_crossfarm_eval_ablation"
OUT.mkdir(parents=True, exist_ok=True)

clip_index = pd.read_csv(FEAT / "clip_feature_index.csv.gz")
labels = pd.read_csv(ROOT / "output" / "11_layer2_proxy_labels" / "proxy_labels_master.csv.gz")
model_base = clip_index[["_record_key"]].drop_duplicates().merge(labels, on="_record_key", how="left")
audio = load_npz(FEAT / "tfidf_1to3_full.npz").tocsr()

CONTEXT_VARS = ["farm", "barn_zone", "microphone", "mic_placement_context", "day", "date"]
meta = clip_index.set_index("_record_key").loc[model_base["_record_key"]].reset_index()
session_group = meta[["farm", "barn_zone", "microphone", "mic_placement_context"]].astype(str).agg("|".join, axis=1).to_numpy()

TARGETS = [("is_nonvocal", "binary"), ("is_distress_related", "binary"),
           ("is_resource_related", "binary"), ("proxy_functional_v1", "multiclass")]


def onehot(cols):
    return OneHotEncoder(handle_unknown="ignore", sparse_output=True).fit_transform(meta[cols].astype(str))


def cv_macro_f1(X, y, task, scheme):
    cv = StratifiedKFold(5, shuffle=True, random_state=42) if scheme == "random" else StratifiedGroupKFold(5)
    splits = cv.split(np.zeros(len(y)), y, session_group) if scheme == "grouped" else cv.split(np.zeros(len(y)), y)
    yt, yp = [], []
    for tr, te in splits:
        solver = "saga" if task == "multiclass" else "liblinear"
        m = LogisticRegression(max_iter=4000, solver=solver, class_weight="balanced", random_state=42)
        m.fit(X[tr], y[tr]); yp += list(m.predict(X[te])); yt += list(y[te])
    return float(f1_score(yt, yp, average="macro", zero_division=0))


rows = []
for target, task in TARGETS:
    y = (LabelEncoder().fit_transform(model_base[target].astype(str)) if task == "multiclass"
         else model_base[target].astype(int).to_numpy())
    for scheme in ["random", "grouped"]:
        base = cv_macro_f1(audio, y, task, scheme)
        rec = {"target": target, "cv": scheme, "audio_only": round(base, 3)}
        for v in CONTEXT_VARS:
            rec[f"gain_{v}"] = round(cv_macro_f1(hstack([audio, onehot([v])], format="csr"), y, task, scheme) - base, 3)
        rec["gain_all"] = round(cv_macro_f1(hstack([audio, onehot(CONTEXT_VARS)], format="csr"), y, task, scheme) - base, 3)
        rows.append(rec)

ablation = pd.DataFrame(rows)
ablation.to_csv(OUT / "context_variable_ablation.csv", index=False)
display(ablation)